This notebook implements a linear regression model from scratch to demonstrate how coefficients are computed from training data. It replicates the class structure and method signatures of scikit-learn’s `LinearRegression` for familiarity and comparability.

In [1]:
import numpy as np
import pandas as pd

In [2]:
import numpy as np

class LinearRegression:
    def __init__(self, fit_intercept=True, learning_rate=0.05, n_iter=1000, random_state=None):
        self.fit_intercept = fit_intercept
        self.learning_rate = learning_rate
        self.n_iter = n_iter
        self.random_state = random_state

        # sklearn-like learned attributes
        self.coef_ = None
        self.intercept_ = None
        self.n_features_in_ = None
        self.feature_names_in_ = None

    def _to_numpy_X(self, X):
        # Keep sklearn-ish behavior: accept array-like / pandas
        if hasattr(X, "to_numpy"):  # pandas DataFrame
            if hasattr(X, "columns"):
                self.feature_names_in_ = np.array(X.columns, dtype=object)
            X = X.to_numpy()
        else:
            X = np.asarray(X)

        if X.ndim != 2:
            raise ValueError(f"X must be 2D (n_samples, n_features). Got shape {X.shape}.")
        return X.astype(float, copy=False)

    def _to_numpy_y(self, y):
        if hasattr(y, "to_numpy"):  # pandas Series/DataFrame
            y = y.to_numpy()
        else:
            y = np.asarray(y)

        # sklearn accepts (n,) or (n,1) for single target
        if y.ndim == 2 and y.shape[1] == 1:
            y = y.ravel()
        if y.ndim != 1:
            raise ValueError(f"y must be 1D for single-target regression. Got shape {y.shape}.")
        return y.astype(float, copy=False)

    def fit(self, X, y):
        X = self._to_numpy_X(X)
        y = self._to_numpy_y(y)

        n_samples, n_features = X.shape
        self.n_features_in_ = n_features

        rng = np.random.default_rng(self.random_state)

        if self.fit_intercept:
            # Add bias column internally, but expose intercept_ separately (like sklearn)
            X_design = np.c_[X, np.ones(n_samples)]
            beta = rng.standard_normal(n_features + 1)
        else:
            X_design = X
            beta = rng.standard_normal(n_features)

        # Gradient descent on MSE = (1/n) * ||y - Xb||^2
        for _ in range(self.n_iter):
            y_pred = X_design @ beta
            grad = -(2 / n_samples) * (X_design.T @ (y - y_pred))
            beta -= self.learning_rate * grad

        if self.fit_intercept:
            self.coef_ = beta[:n_features]
            self.intercept_ = beta[-1]
        else:
            self.coef_ = beta
            self.intercept_ = 0.0

        return self

    def predict(self, X):
        if self.coef_ is None:
            raise ValueError("This LinearRegression instance is not fitted yet. Call 'fit' first.")

        if hasattr(X, "to_numpy"):
            X = X.to_numpy()
        else:
            X = np.asarray(X)

        if X.ndim != 2:
            raise ValueError(f"X must be 2D (n_samples, n_features). Got shape {X.shape}.")
        if self.n_features_in_ is not None and X.shape[1] != self.n_features_in_:
            raise ValueError(f"X has {X.shape[1]} features, but model was fit with {self.n_features_in_}.")

        X = X.astype(float, copy=False)
        return X @ self.coef_ + self.intercept_


Usage

In [3]:
X = pd.DataFrame({"A":[1,2,3,4], "B":[4,5,6,8]})
y = pd.Series([2,5,3,4])

lr = LinearRegression(fit_intercept=True, learning_rate=0.01, n_iter=5000, random_state=0)
lr.fit(X, y)

print(lr.coef_, lr.intercept_)
pred = lr.predict(X)


[0.35122079 0.04760046] 2.340116341403909
